In [3]:
from pyspark.sql import SparkSession
import socket 

# 1. Master URL (using spark protocol)
master_url = "spark://localhost:7077"

# 2. Driver Host IP (Laptop's IP on the Docker bridge network)
driver_host_ip = "172.26.0.1" 

print(f"Attempting to connect to master at: {master_url}")
print(f"Will announce driver host IP as: {driver_host_ip}")
# 
# Optional check (runs on your laptop, not inside Docker)
try:
    socket.gethostbyname(driver_host_ip)
    print(f"Check: IP {driver_host_ip} resolves locally.")
except socket.gaierror:
    print(f"Warning: IP {driver_host_ip} might not resolve directly on the host machine, this is usually OK.")

# 3. Build SparkSession with driver host configuration
try:
    spark = SparkSession.builder \
        .master(master_url) \
        .appName("My First Distributed Test") \
        .config("spark.driver.host", driver_host_ip) \
        .getOrCreate()

    # Get the SparkContext
    sc = spark.sparkContext

    print("\n--- Connection Successful! --- ✅")
    print(f"SparkSession object: {spark}")
    print(f"SparkContext object: {sc}")
    print(f"Spark version in use: {sc.version}") # Should confirm 4.0.1

except Exception as e:  
    print(f"\n--- Connection Failed --- ❌")
    print(f"Error details: {e}")

Attempting to connect to master at: spark://localhost:7077
Will announce driver host IP as: 172.26.0.1
Check: IP 172.26.0.1 resolves locally.

--- Connection Successful! --- ✅
SparkSession object: <pyspark.sql.session.SparkSession object at 0x7f47b6a05360>
SparkContext object: <SparkContext master=spark://localhost:7077 appName=My First Distributed Test>
Spark version in use: 4.0.1


In [4]:
import time
import datetime

# Number of "tasks" (simulating images)
num_tasks = 1000 
# Time to simulate processing of each task (in seconds)
time_per_task = 0.1 

# --- Distributed code with PySpark ---

print(f"--- Starting distributed test with {num_tasks} tasks ---")

# Initial RDD 
rdd = sc.parallelize(range(num_tasks), numSlices=100)

# Function that simulates a slow task
def slow_task(number):
    time.sleep(time_per_task)
    return number * 2

start_time_spark = datetime.datetime.now()

counted_items = rdd.map(slow_task).count()

end_time_spark = datetime.datetime.now()
duration_spark = end_time_spark - start_time_spark

print(f"\n--- Distributed test completed ---")
print(f"Total items counted: {counted_items}")
print(f"Execution time (Spark): {duration_spark}")

# Expected (theoretical, without overhead): (num_tasks * time_per_task) / total_cores
# total_cores = 3 workers * 2 cores/worker = 6
total_cores = 6
expected_theoretical_time = (num_tasks * time_per_task) / total_cores
print(f"Ideal theoretical time (without overhead): {expected_theoretical_time:.2f} seconds")

--- Starting distributed test with 1000 tasks ---



--- Distributed test completed ---
Total items counted: 1000
Execution time (Spark): 0:00:20.930533
Ideal theoretical time (without overhead): 16.67 seconds


In [5]:
import datetime

# Same parameters as before
num_tasks = 1000
time_per_task = 0.1

# --- Sequential Code with Pure Python ---

print(f"--- Starting sequential test with {num_tasks} tasks ---")

sequential_results = []

# Mark start time
start_time_seq = datetime.datetime.now()

# Traditional for loop
for i in range(num_tasks):
    # Apply the same work function
    result = slow_task(i) 
    sequential_results.append(result)

# Mark end time
end_time_seq = datetime.datetime.now()
duration_seq = end_time_seq - start_time_seq

print(f"\n--- Sequential test completed ---")
print(f"Total elements processed: {len(sequential_results)}")
print(f"Execution time (Sequential): {duration_seq}")

# Expected (theoretical): num_tasks * time_per_task
expected_sequential_time = num_tasks * time_per_task
print(f"Theoretical expected time: {expected_sequential_time:.2f} seconds")

--- Starting sequential test with 1000 tasks ---

--- Sequential test completed ---
Total elements processed: 1000
Execution time (Sequential): 0:01:40.255745
Theoretical expected time: 100.00 seconds

--- Sequential test completed ---
Total elements processed: 1000
Execution time (Sequential): 0:01:40.255745
Theoretical expected time: 100.00 seconds


In [ ]:
spark.stop()